# oold AI playground

An LLM agent in the left sidebar and a Pyodide kernel behind it. The agent runs Python
through the `jupyterlab-ai-commands:execute-in-kernel` command, which uses the kernel of
this notebook, so everything it does lands here and you can inspect it from these cells.

Nothing leaves the browser except the calls to your model provider. There is no server:
the kernel, the files and the schemas live on the contents drive at `/drive`.

## Before the first message

Open **AI Settings** from the chat toolbar and add a provider with its API key. That is
the only step.

The system prompt is already filled in. The site ships `OOLD-AGENT.md` as the default, so
the agent arrives knowing the Python API, what a check id means, and that this kernel has
no shell. The file sits next to this notebook; edit the prompt in **AI Settings** to change
what the agent is told.

## Bootstrap

Two lines. `oold_agent` has to be imported before anything touches `oold`, because it sets
`OOLD_DESCRIPTOR_BINDING` at import time. `setup` installs the wheels into the kernel and is
the only coroutine in the module.

In [ ]:
import oold_agent

await oold_agent.setup()
print("AI PLAYGROUND READY")

## First instruction

Send this to the agent:

> Profile `fixture_measurements.csv`, draft an OO-LD schema for it, validate the draft
> before you write it to disk, and report the check id of anything that fails.

Spreadsheets work the same way. `xlsx_profile(path, sheet=None)` returns what
`csv_profile` returns plus `sheet` and `sheets`, and `sheets(path)` lists the sheet names
on their own.

Documents too. `doc_profile(path, table=None)` profiles one table out of a docx, a pptx or
a pdf into that same column shape, `tables(path)` lists the tables, and `doc_text(path)`
returns the prose around them: the outline, the text per paragraph, slide or page, and all
of it joined. The format is read from the bytes, so a file whose suffix lies still opens.

Fixtures on the drive to work against:

| file | what it is |
| --- | --- |
| `fixture_measurements.csv` | four rows, one identifier column, one column with nulls |
| `fixture_malformed.csv` | duplicate header and ragged rows |
| `fixture_measurements.xlsx` | the same four rows as the CSV, one sheet |
| `fixture_workbook.xlsx` | three sheets, one of them empty |
| `fixture_merged_header.xlsx` | a merged title row above the real header |
| `fixture_legacy.xls` | legacy BIFF, which calamine opens as well |
| `fixture_corrupt.xlsx` | a truncated zip, reported as a problem rather than raised |
| `fixture_protocol.docx` | two tables under two headings, one a data dictionary |
| `fixture_notes.docx` | text and no table |
| `fixture_overview.pptx` | three slides, a table on the last |
| `fixture_report.pdf` | text on page one, a ruled table on page two |
| `fixture_mislabelled.docx` | a pdf under a docx suffix, read as what it is |
| `fixture_Measurement.schema.json` | a schema that passes |
| `fixture_Broken.schema.json` | fails `schema.meta` and `context.predicates` |
| `fixture_Measurement.instance.json` | an instance that validates |
| `fixture_bad.instance.json` | fails `instance.schema` |

To check the toolchain by hand rather than through the agent:

In [ ]:
report = oold_agent.validate_schema("fixture_Broken.schema.json")
print(report.passed, report.counts)
for check in report.failures():
    print(check.id, check.rule, check.message)

## GitHub

`oold_git` reaches the GitHub REST API from the kernel. Reads need no token, writes need a
fine-grained PAT. Always pass `paths`: without it the whole tree is fetched at one request
per file and unauthenticated GitHub allows 60 per hour.

```python
import piplite
await piplite.install(["requests", "pyodide-http"])
import oold_git

oold_git.pull("SINTEF/physmet-data-documentation-templates", into="gh",
              paths=["shared/*.csv", "context/context.json"])
```